In [12]:
import inspect
import sys
from typing import Any, Callable, Generic, TypeVar
from makefun import wraps as mf_wraps
from plum import dispatch, parametric, type_parameter
from beartype.peps import resolve_pep563

# ==========================================
# 1. TARGET DATA STRUCTURE
# ==========================================
class COO:
    """The canonical Coordinate List target format."""
    def __init__(self, elements_count: int):
        self.elements_count = elements_count
        
    def __repr__(self):
        return f"COO(elements={self.elements_count})"


# ==========================================
# 2. THE PARAMETRIC TRAIT / TYPECLASS
# ==========================================
T = TypeVar('T')

@parametric
class SupportsCOO(Generic[T]):
    """
    A parametric typeclass container.
    Serves as the dispatch target for converting arbitrary types to COO.
    """
    def __init__(self, value: T):
        self.value = value

    @classmethod
    def convert(cls, obj: Any) -> COO:
        """
        Dynamically locates the concrete parametric typeclass variant 
        (e.g., SupportsCOO[CSRMatrix]) and executes its conversion routine.
        """
        concrete_typeclass = SupportsCOO[type(obj)]
        return concrete_typeclass._to_coo(obj)

    @classmethod
    def _to_coo(cls, obj: Any) -> COO:
        """
        FALLBACK METHOD
        Executes if no specialized @dispatch overload exists for the incoming type.
        """
        # Look for standard duck-typed serialization methods
        if hasattr(obj, "to_coo"):
            print("-> Found fallback duck-typed .to_coo() method")
            return obj.to_coo()
            
        raise NotImplementedError(
            f"Type '{type(obj).__name__}' does not support conversion to COO. "
            f"Register it via @dispatch on SupportsCOO[{type(obj).__name__}]."
        )


# ==========================================
# 3. FLATTENED DECORATOR ENGINE
# ==========================================
def implements_coo(func: Callable) -> Callable:
    """
    A flattened, two-level signature-preserving decorator.
    Intercepts variables annotated as SupportsCOO and normalizes them to COO.
    """
    # Fix potential PEP-563 string annotation evaluation edge cases
    resolve_pep563(func)
    
    # Track down parameters wrapped in the typeclass constraint
    marked_parameters = set()
    for param_name, annotation in func.__annotations__.items():
        if hasattr(annotation, '__origin__') and annotation.__origin__ is SupportsCOO:
            marked_parameters.add(param_name)

    @mf_wraps(func)
    def wrapper(*args: Any, **kwargs: Any) -> Any:
        sig = inspect.signature(func)
        bound = sig.bind(*args, **kwargs)
        bound.apply_defaults()
        
        # Coerce structural types at runtime
        for param_name in list(bound.arguments.keys()):
            if param_name in marked_parameters:
                raw_val = bound.arguments[param_name]
                
                if isinstance(raw_val, COO):
                    bound.arguments[param_name] = raw_val
                else:
                    bound.arguments[param_name] = SupportsCOO.convert(raw_val)
                    
        # Guarantee positional-to-keyword conversion consistency
        return func(**bound.arguments)
        
    return wrapper


# ==========================================
# 4. INTROSPECTION ENGINE (AUTO-DOCS)
# ==========================================
def get_supported_coo_types() -> list[str]:
    """
    Queries Plum's global method registry to find all explicitly 
    registered overloads under the SupportsCOO typeclass namespace.
    """
    supported = []
    # Fetch the multi-dispatch function tracking system for _to_coo
    # It lives inside the class namespace as a Plum Function object
    for method in SupportsCOO._to_coo.methods:
        if method.signature.types:
            # Grab the class parameter constraint (e.g., type[SupportsCOO[CSRMatrix]])
            meta_cls = method.signature.types[0]
            # Dig out the inner type parameter if it's a parametric variant
            if hasattr(meta_cls, "__args__") and hasattr(meta_cls.__args__[0], "__args__"):
                source_type = meta_cls.__args__[0].__args__[0]
                if source_type is not Any:
                    supported.append(source_type.__name__)
    return list(set(supported))


# ==========================================
# 5. INPUT DATA FORMAT OVERLOADS
# ==========================================
class CSRMatrix: pass
class ScipyGraph: pass

class CustomDuckType:
    """An external object providing native conversion compatibility."""
    def to_coo(self): 
        return COO(999)

# Register specialized traits dynamically across codebases via global dispatch
@dispatch
def _to_coo(cls: type[SupportsCOO[CSRMatrix]], obj: CSRMatrix) -> COO:
    print("-> Processing specialized CSRMatrix pipeline path")
    return COO(500)

@dispatch
def _to_coo(cls: type[SupportsCOO[ScipyGraph]], obj: ScipyGraph) -> COO:
    print("-> Processing specialized ScipyGraph pipeline path")
    return COO(1200)


 

In [13]:
Supports[COO]

__main__.Supports[__main__.COO]

In [14]:

@implements_coo
def compute_sparse_metrics(data: SupportsCOO):
    """Expects anything supporting COO. 'data' arrives fully materialized as a COO object."""
    return f"Pipeline successfully read elements_count: {data.elements_count}"


print("--- 🔍 Live Auto-Doc Extraction ---")
print(f"Registered conversion types: {get_supported_coo_types()}")
# Output: ['CSRMatrix', 'ScipyGraph']

print("\n--- ⚡ Executing Specialized Path ---")
print(compute_sparse_metrics(CSRMatrix()))
# Output:
# -> Processing specialized CSRMatrix pipeline path
# Pipeline successfully read elements_count: 500

print("\n--- ⚡ Executing Duck-Type Fallback ---")
print(compute_sparse_metrics(CustomDuckType()))
# Output:
# -> Found fallback duck-typed .to_coo() method
# Pipeline successfully read elements_count: 999

print("\n--- 📜 Signature Verification ---")
print(inspect.signature(compute_sparse_metrics))
# Output: (data: __main__.SupportsCOO)


--- 🔍 Live Auto-Doc Extraction ---


AttributeError: 'function' object has no attribute 'methods'

Bad pipe message: %s [b'\'"\xfa\xf1\x10e`d\x94\xac\xd8#\xc3\xbf\x99\x9el\xe9 <\x10\x8fI\x82iR\x078~\xbb\xf8|F\x93\x931U).\xc6\xa2\x05\xb8x\'*\x9f\xa1^\xe6\x95\x00\x06\x13\x02\x13\x01\x13\x03\x01\x00\x0bI\x00\x00\x00\n\x00\x08\x00\x00\x05MODIV\x00+\x00\x03\x02\x03\x04\x00\n\x00\x0c\x00\n\x11\xec\x02\x01\x00\x1d\x00\x17\x00\x18\x003\n8\n6\x11\xec\x04\xc0\x0b;\x1d=\x10z\x9d\xack\xb8:+\x9e\xcb\xc2\xc0\x91\x90\xf4\xfb\xc7\xf1&" 5\xa6\xee\xa7A\x92\xf7\x9e\xaa[!\xafBp\xa2P\x8b\xaa\x83.\xbb8P\xd0\xb7*\n\xd1r\xeeT\xa8\xa5\xdaC\xba\xf4\x89\xc3\x81B\x1aZ\x9d<X\n\x8f#\xb1F<_\xaa\xf3H\xb8\x85\x87S\xc2!^\xd0]\xa6\xa8\'^\xe8;\xabk\x00N\x1c\x11TXX\xf3F\x9e_eGlR\xa8R\xebUp', b'Fa\xce\xb3I;\xeeB\xb4\x93 \xcc\xcc\x9a\xb6\xcd\\^\xe5*\x06\xe2Y\xc1\xd6\xc3\x84\xb0\x06\xca\xf2q\x01W\tZ\xea\x162\xe3\x9a\x05w\x81\x01\xe6\x1a\xae\xa7X<\x1f\xb1\x0e\x83sr\xb3\x059\xd6HPw\xd5\xb9\xc5K\xc2$\xb4\x9c\xb2\xf1\xb9k\xe1\r\x16`\x9f\xa3\x10\xcf\xe1 \x8fb\x8bZS\x17D\xed\xe9&\x1d\xb1@\x136q\x0f\x98\x88\xfd%\x16Ca\x88\xe98\x

In [4]:
Dispatcher.functions

<attribute 'functions' of 'Dispatcher' objects>

In [15]:
from typing import Any, Callable, Type
from makefun import wraps
from plum import convert, conversion_method
from beartype.peps import resolve_pep563
from beartype import beartype

def normalize_with_plum_simple(target_type: Type) -> Callable:
    """
    A significantly simpler decorator that uses beartype to resolve annotations,
    and makefun.wraps to cleanly handle signature copying without manual signature extraction.
    """
    def decorator(func: Callable) -> Callable:
        # 1. Mutate func.__annotations__ in place to resolve string evaluations
        resolve_pep563(func)
        
        # 2. Use makefun.wraps directly on the inner wrapper
        @wraps(func)
        def wrapper(*args: Any, **kwargs: Any) -> Any:
            converted_args = tuple(convert(arg, target_type) for arg in args)
            converted_kwargs = {k: convert(v, target_type) for k, v in kwargs.items()}
            return func(*converted_args, **converted_kwargs)
            
        return wrapper
        
    return decorator

# --- Example Usage ---

class TargetType:
    def __init__(self, value: Any):
        self.value = value
    def __repr__(self):
        return f"TargetType({self.value!r})"

@conversion_method(type_from=str, type_to=TargetType)
def _str_to_target(obj: str) -> TargetType:
    return TargetType(f"Parsed: {obj}")

@conversion_method(type_from=int, type_to=TargetType)
def _str_to_target(obj: int) -> TargetType:
    return TargetType(f"Parsed: {obj}")


@normalize_with_plum_simple(TargetType)
def process_data(a: str|int, b: str|int) -> str:
    """Combines two data fields smoothly."""
    print("combining values", a.value, b.value) 
    return f"{a} & {b}"

# Test execution
print(process_data(0, "world"))
# Output: TargetType('Parsed: hello') & TargetType('Parsed: world')

# Introspect via standard tools to confirm signature preservation
from inspect import signature
print("Preserved Signature:", signature(process_data))

# print(process_data(b'10', 2.0))
# Output: Preserved Signature: (a: str, b: str) -> str


combining values Parsed: 0 Parsed: world
TargetType('Parsed: 0') & TargetType('Parsed: world')
Preserved Signature: (a: str | int, b: str | int) -> str


In [16]:
from typing import Any, Callable, Generic, Type, TypeVar, get_args
from makefun import wraps
from plum import convert, conversion_method
from beartype.peps import resolve_pep563

# 1. Define a Generic "Supports" Typeclass Marker
T = TypeVar('T')

class Supports(Generic[T]):
    """Statically hints that a type can be converted to T via Plum."""
    pass

def typeclass_normalizer() -> Callable:
    """
    A decorator that looks at the function's type hints for Supports[X],
    and automatically normalizes those specific arguments using Plum.
    """
    def decorator(func: Callable) -> Callable:
        resolve_pep563(func)
        
        # Introspect the signature to see WHICH arguments require conversion
        # maps argument name -> the target type it needs to become
        conversion_map: dict[str, Type] = {}
        
        for param_name, annotation in func.__annotations__.items():
            # Check if the annotation is a structural 'Supports[Target]' type
            if hasattr(annotation, '__origin__') and annotation.__origin__ is Supports:
                target_type = get_args(annotation)[0]
                conversion_map[param_name] = target_type

        @wraps(func)
        def wrapper(*args: Any, **kwargs: Any) -> Any:
            # Re-map positional arguments based on function signature position
            # (Simplifying here by converting kwargs for explicit parameter naming)
            # A robust implementation maps args to their parameter names:
            import inspect
            sig = inspect.signature(func)
            bound = sig.bind(*args, **kwargs)
            bound.apply_defaults()
            
            # Apply Plum conversion ONLY to parameters marked with Supports[...]
            for param_name, value in bound.arguments.items():
                if param_name in conversion_map:
                    target = conversion_map[param_name]
                    bound.arguments[param_name] = convert(value, target)
                    
            return func(*bound.args, **bound.kwargs)
            
        return wrapper
    return decorator

# --- Example Usage ---

class TargetType:
    def __init__(self, value: Any):
        self.value = value
    def __repr__(self):
        return f"TargetType({self.value!r})"

# Register custom converters anywhere in your app at any time
@conversion_method(type_from=str, type_to=TargetType)
def _str_to_target(obj: str) -> TargetType:
    return TargetType(f"From Str: {obj}")

@conversion_method(type_from=int, type_to=TargetType)
def _int_to_target(obj: int) -> TargetType:
    return TargetType(obj * 100)


# 2. Write your function with the clean Typeclass signature!
@typeclass_normalizer()
def process_data(a: Supports[TargetType], b: Supports[TargetType]) -> str:
    """
    Statically accepts anything that can convert to TargetType.
    At runtime, 'a' and 'b' arrive fully converted to TargetType.
    """
    return f"Processed: {a} and {b}"

# You can pass strings or ints seamlessly!
print(process_data("hello", 5))
# Output: Processed: TargetType('From Str: hello') and TargetType(500)

# Adding a completely new type later requires ZERO changes to process_data
@conversion_method(type_from=float, type_to=TargetType)
def _float_to_target(obj: float) -> TargetType:
    return TargetType(f"From Float: {obj}")
    
print(process_data(3.14, "world"))
# Output: Processed: TargetType('From Float: 3.14') and TargetType('From Str: world')


Processed: TargetType('From Str: hello') and TargetType(500)
Processed: TargetType('From Float: 3.14') and TargetType('From Str: world')


In [17]:
process_data

<function __main__.process_data(a: __main__.Supports[__main__.TargetType], b: __main__.Supports[__main__.TargetType]) -> str>

In [18]:
Supports[TargetType]

__main__.Supports[__main__.TargetType]

In [19]:
process_data(b'10', r'10')

TypeError: Cannot convert `b'10'` to `__main__.TargetType`.

In [21]:
from typing import Any, Callable, Generic, Type, TypeVar
from makefun import wraps
from plum import convert, conversion_method
from beartype.peps import resolve_pep563

T = TypeVar('T')

class Supports(Generic[T]):
    pass

# ==========================================
# 🔍 THE DESIRED INTROSPECTION RUNTIME TOOL
# ==========================================
def get_supported_conversions(target_type: Type) -> list[Type]:
    """
    Introspects Plum's live global dispatch registry at runtime to automatically 
    extract all source types that can currently be converted into the target_type.
    """
    supported_sources: list[Type] = []
    
    # plum.convert stores its registered conversion overloads inside .methods
    for method in convert.methods:
        # Each method signature has a list of type parameters.
        # Conversion signatures match: (instance_type, Type[target_type])
        types = method.signature.types
        
        if len(types) >= 2:
            source_candidate = types[0]
            target_candidate_type_wrapper = types[1]
            
            # Plum wraps target constraints in Type[X]. We extract the raw inner class.
            # e.g., extracting TargetType out of Type[TargetType]
            if hasattr(target_candidate_type_wrapper, "__args__"):
                actual_target = target_candidate_type_wrapper.__args__[0]
                
                if actual_target is target_type:
                    supported_sources.append(source_candidate)
                    
    return supported_sources

# ==========================================
# The Decorator / Typeclass implementation
# ==========================================
def implements(target_type: Type) -> Callable:
    def decorator(func: Callable) -> Callable:
        resolve_pep563(func)
        
        # Inject dynamic docstring auto-generation right at definition time!
        sources = get_supported_conversions(target_type)
        source_names = ", ".join([getattr(s, "__name__", str(s)) for s in sources])
        
        func.__doc__ = (func.__doc__ or "") + f"\n\n[Typeclass: Supports implementations from: {source_names}]"

        @wraps(func)
        def wrapper(*args: Any, **kwargs: Any) -> Any:
            import inspect
            sig = inspect.signature(func)
            bound = sig.bind(*args, **kwargs)
            bound.apply_defaults()
            
            # Simple conversion hook
            for param_name, value in bound.arguments.items():
                annotation = func.__annotations__.get(param_name)
                if hasattr(annotation, '__origin__') and annotation.__origin__ is Supports:
                    bound.arguments[param_name] = convert(value, target_type)
                    
            return func(*bound.args, **bound.kwargs)
        return wrapper
    return decorator

# --- Example Usage ---

class JSONPayload:
    def __init__(self, value: str):
        self.value = value
    def __repr__(self):
        return f"JSONPayload({self.value})"

# 1. Base registrations
@conversion_method(type_from=int, type_to=JSONPayload)
def _int_to_json(obj: int) -> JSONPayload: return JSONPayload(str(obj))

@conversion_method(type_from=dict, type_to=JSONPayload)
def _dict_to_json(obj: dict) -> JSONPayload: return JSONPayload(str(obj))


@implements(JSONPayload)
def process(data: Supports[JSONPayload]):
    """Processes pipeline data safely."""
    return data


# Check the docstring generated automatically at initialization
print("--- Docstring at Launch ---")
print(process.__doc__)

print("\n--- Live Querying Runtime Plugins ---")
# Imagine a user imports a third-party plugin right now that adds a new conversion:
@conversion_method(type_from=str, type_to=JSONPayload)
def _str_to_json(obj: str) -> JSONPayload: return JSONPayload(f'"{obj}"')

# Querying live tells your documentation generator or API schema builder 
# exactly what is possible *right now* dynamically!
current_conversions = get_supported_conversions(JSONPayload)
print("Currently supported conversions for JSONPayload:")
for src in current_conversions:
    print(f" - {src.__name__}")


--- Docstring at Launch ---
Processes pipeline data safely.

[Typeclass: Supports implementations from: ]

--- Live Querying Runtime Plugins ---
Currently supported conversions for JSONPayload:


In [26]:
convert.methods[0from typing import Any, Callable, Generic, Type, TypeVar
from makefun import wraps
from plum import convert, conversion_method
from beartype.peps import resolve_pep563

# Import Plum's actual underlying conversion registry object
from plum.conversion import conversion_registry

T = TypeVar('T')

class Supports(Generic[T]):
    pass

# ==================================================
# 🔍 THE CORRECTED INTROSPECTION RUNTIME TOOL
# ==================================================
def get_supported_conversions(target_type: Type) -> list[Type]:
    """
    Introspects Plum's internal conversion registry to dynamically extract 
    all source types that can currently be converted into the target_type.
    """
    supported_sources: list[Type] = []
    
    # conversion_registry is a dictionary mapping: (type_from, type_to) -> conversion_function
    for (type_from, type_to), _ in conversion_registry.items():
        if type_to is target_type:
            supported_sources.append(type_from)
            
    return supported_sources

# ==================================================
# The Refactored Traits/Typeclass Decorator
# ==================================================
def implements(target_type: Type) -> Callable:
    def decorator(func: Callable) -> Callable:
        resolve_pep563(func)

        @wraps(func)
        def wrapper(*args: Any, **kwargs: Any) -> Any:
            import inspect
            sig = inspect.signature(func)
            bound = sig.bind(*args, **kwargs)
            bound.apply_defaults()
            
            for param_name, value in bound.arguments.items():
                annotation = func.__annotations__.get(param_name)
                if hasattr(annotation, '__origin__') and annotation.__origin__ is Supports:
                    bound.arguments[param_name] = convert(value, target_type)
                    
            return func(*bound.args, **bound.kwargs)
        return wrapper
    return decorator

# --- Example Usage & Dynamic Introspection Verification ---

class JSONPayload:
    def __init__(self, value: str):
        self.value = value
    def __repr__(self):
        return f"JSONPayload({self.value})"

# 1. Base registrations before building documentation
@conversion_method(type_from=int, type_to=JSONPayload)
def _int_to_json(obj: int) -> JSONPayload: return JSONPayload(str(obj))

@conversion_method(type_from=dict, type_to=JSONPayload)
def _dict_to_json(obj: dict) -> JSONPayload: return JSONPayload(str(obj))


@implements(JSONPayload)
def process(data: Supports[JSONPayload]):
    """Processes pipeline data safely."""
    return data


if __name__ == "__main__":
    print("--- Initial Querying ---")
    initial_conversions = get_supported_conversions(JSONPayload)
    print("Initial types supported for JSONPayload:")
    for src in initial_conversions:
        print(f" - {src.__name__}")
    # Output properly includes: int, dict
    
    print("\n--- Simulating a Runtime Plugin Drop ---")
    # A user dynamically drops a brand new conversion from an external file:
    @conversion_method(type_from=str, type_to=JSONPayload)
    def _str_to_json(obj: str) -> JSONPayload: return JSONPayload(f'"{obj}"')
    
    # Querying the registry again immediately reflects the new plugin
    updated_conversions = get_supported_conversions(JSONPayload)
    print("Updated types supported for JSONPayload:")
    for src in updated_conversions:
        print(f" - {src.__name__}")
    # Output now perfectly catches: int, dict, str
]

convert(obj: object, type_to: Any)
    <function convert at 0x75ba2c40fe20> @ 
]8;id=3639977;file:///home/rtbs/syncthing/nist-docs/code/thesis/affinis/.venv/lib/python3.12/site-packages/plum/_promotion.py#35\~/syncthing/nist-docs/code/thesis/affinis/.venv/lib/python3.12/site-packages/plum/]8;;\]8;id=3639978;file:///home/rtbs/syncthing/nist-docs/code/thesis/affinis/.venv/lib/python3.12/site-packages/plum/_promotion.py#35\_promotion.py]8;;\]8;id=3639977;file:///home/rtbs/syncthing/nist-docs/code/thesis/affinis/.venv/lib/python3.12/site-packages/plum/_promotion.py#35\:35]8;;\

In [28]:
from typing import Any, Callable, Generic, Type, TypeVar
from makefun import wraps
from plum import convert, conversion_method
from beartype.peps import resolve_pep563

# Import Plum's actual underlying conversion registry object
from plum import 

T = TypeVar('T')

class Supports(Generic[T]):
    pass

# ==================================================
# 🔍 THE CORRECTED INTROSPECTION RUNTIME TOOL
# ==================================================
def get_supported_conversions(target_type: Type) -> list[Type]:
    """
    Introspects Plum's internal conversion registry to dynamically extract 
    all source types that can currently be converted into the target_type.
    """
    supported_sources: list[Type] = []
    
    # conversion_registry is a dictionary mapping: (type_from, type_to) -> conversion_function
    for (type_from, type_to), _ in conversion_registry.items():
        if type_to is target_type:
            supported_sources.append(type_from)
            
    return supported_sources

# ==================================================
# The Refactored Traits/Typeclass Decorator
# ==================================================
def implements(target_type: Type) -> Callable:
    def decorator(func: Callable) -> Callable:
        resolve_pep563(func)

        @wraps(func)
        def wrapper(*args: Any, **kwargs: Any) -> Any:
            import inspect
            sig = inspect.signature(func)
            bound = sig.bind(*args, **kwargs)
            bound.apply_defaults()
            
            for param_name, value in bound.arguments.items():
                annotation = func.__annotations__.get(param_name)
                if hasattr(annotation, '__origin__') and annotation.__origin__ is Supports:
                    bound.arguments[param_name] = convert(value, target_type)
                    
            return func(*bound.args, **bound.kwargs)
        return wrapper
    return decorator

# --- Example Usage & Dynamic Introspection Verification ---

class JSONPayload:
    def __init__(self, value: str):
        self.value = value
    def __repr__(self):
        return f"JSONPayload({self.value})"

# 1. Base registrations before building documentation
@conversion_method(type_from=int, type_to=JSONPayload)
def _int_to_json(obj: int) -> JSONPayload: return JSONPayload(str(obj))

@conversion_method(type_from=dict, type_to=JSONPayload)
def _dict_to_json(obj: dict) -> JSONPayload: return JSONPayload(str(obj))


@implements(JSONPayload)
def process(data: Supports[JSONPayload]):
    """Processes pipeline data safely."""
    return data


print("--- Initial Querying ---")
initial_conversions = get_supported_conversions(JSONPayload)
print("Initial types supported for JSONPayload:")
for src in initial_conversions:
    print(f" - {src.__name__}")
# Output properly includes: int, dict

print("\n--- Simulating a Runtime Plugin Drop ---")
# A user dynamically drops a brand new conversion from an external file:
@conversion_method(type_from=str, type_to=JSONPayload)
def _str_to_json(obj: str) -> JSONPayload: return JSONPayload(f'"{obj}"')

# Querying the registry again immediately reflects the new plugin
updated_conversions = get_supported_conversions(JSONPayload)
print("Updated types supported for JSONPayload:")
for src in updated_conversions:
    print(f" - {src.__name__}")
# Output now perfectly catches: int, dict, str


ImportError: cannot import name 'conversion_registry' from 'plum' (/home/rtbs/syncthing/nist-docs/code/thesis/affinis/.venv/lib/python3.12/site-packages/plum/__init__.py)